
# Baseline de référence - Prédiction de la volatilité du Bitcoin

## Projet VolatiChainXplorerAI - Sprint 2

### Objectif

Implémenter et évaluer un modèle de référence fondé sur la persistance de la volatilité historique, pour les horizons de 7 et 30 jours.

### Principe

La volatilité historique observée sur une période donnée est utilisée comme estimation de la volatilité future sur une période de même durée.

- Horizon 7 jours : `volatility_7d` -> `target_volatility_7d`
- Horizon 30 jours : `volatility_30d` -> `target_volatility_30d`

### Protocole d'évaluation

1. Réutiliser les partitions chronologiques de validation et de test déjà constituées.
2. Appliquer la règle de persistance sans entraînement.
3. Calculer les métriques MAE, MSE, RMSE et R² sur la validation.
4. Appliquer la même règle sur le test final, sans modification.
5. Exporter les résultats pour l'analyse comparative avec PyCaret et AutoGluon.

Aucune modification des datasets existants n'est nécessaire.


## 1. Initialisation

### Chemin vers les dosiers des partitions

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Racine du projet
project_root = Path.cwd().parent

# Répertoire des partitions communes
splits_dir = (
    project_root 
    / "data" 
    / "exports" 
    / "datasets" 
    /"benchmark_splits"
)

print("Racine du projet :", project_root)
print("Dossier des partitions présent :", splits_dir.is_dir())

Racine du projet : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI
Dossier des partitions présent : True


### Vérifier les fichiers disponibles

In [3]:
# Fichiers nécessaires à l'évaluation de la baseline
required_files = [
    "validation_7d.csv",
    "test_7d.csv",
    "validation_30d.csv",
    "test_30d.csv"
]

for filename in required_files:
    filepath = splits_dir / filename
    print(f"{filename} : {'OK' if filepath.is_file() else 'INTROUVABLE'}")

validation_7d.csv : OK
test_7d.csv : OK
validation_30d.csv : OK
test_30d.csv : OK


## 2. Charger et vérifier les données

In [4]:
# Chargement des partitions de validation et de test
validation_7d = pd.read_csv(splits_dir / "validation_7d.csv")
test_7d = pd.read_csv(splits_dir / "test_7d.csv")

validation_30d = pd.read_csv(splits_dir / "validation_30d.csv")
test_30d = pd.read_csv(splits_dir / "test_30d.csv")

# Vérification des dimensions et des colonnes nécessaires
datasets = {
    "Validation 7j": (validation_7d, "volatility_7d", "target_volatility_7d"),
    "Test 7j": (test_7d, "volatility_7d", "target_volatility_7d"),
    "Validation 30j": (validation_30d, "volatility_30d", "target_volatility_30d"),
    "Test 30j": (test_30d, "volatility_30d", "target_volatility_30d")
}

for name, (df, feature, target) in datasets.items():
    print(f"\n{name} : {df.shape}")
    print("Volatilité historique présente :", feature in df.columns)
    print("Cible future présente :", target in df.columns)

    if feature in df.columns and target in df.columns:
        print("Valeurs manquantes :", df[[feature, target]].isna().sum().sum())


Validation 7j : (724, 17)
Volatilité historique présente : True
Cible future présente : True
Valeurs manquantes : 0

Test 7j : (590, 17)
Volatilité historique présente : True
Cible future présente : True
Valeurs manquantes : 0

Validation 30j : (701, 17)
Volatilité historique présente : True
Cible future présente : True
Valeurs manquantes : 0

Test 30j : (590, 17)
Volatilité historique présente : True
Cible future présente : True
Valeurs manquantes : 0


## 3. Appliquer la baseline sur les données de validation

La baseline repose sur un modèle de persistance : la volatilité historique observée sert directement de prédiction de la volatilité future.

- **Horizon 7 jours** : prédiction = `volatility_7d`
- **Horizon 30 jours** : prédiction = `volatility_30d`

Les prédictions sont comparées aux cibles futures correspondantes.

Cette méthode ne nécessite aucun entraînement et n'utilise aucune information future pour produire les prédictions.


In [5]:
# Baseline de persistance - Validation 7 jours
y_val_7d = validation_7d["target_volatility_7d"]
y_pred_val_7d = validation_7d["volatility_7d"]

# Baseline de persistance - Validation 30 jours
y_val_30d = validation_30d["target_volatility_30d"]
y_pred_val_30d = validation_30d["volatility_30d"]

# Vérification
print("Validation 7j :", len(y_val_7d), "observations")
print("Prédictions 7j :", len(y_pred_val_7d))

print("Validation 30j :", len(y_val_30d), "observations")
print("Prédictions 30j :", len(y_pred_val_30d))

Validation 7j : 724 observations
Prédictions 7j : 724
Validation 30j : 701 observations
Prédictions 30j : 701


## 4. Calculer les métriques de validation

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Fonction commune d'évaluation
def evaluate_baseline(y_true, y_pred, horizon):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    return {
        "horizon": horizon,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

# Évaluation sur les données de validation
validation_results = pd.DataFrame([
    evaluate_baseline(y_val_7d, y_pred_val_7d, "7d"),
    evaluate_baseline(y_val_30d, y_pred_val_30d, "30d")
])

display(validation_results)

,horizon,MAE,MSE,RMSE,R2
0,7d,0.010130,0.000173,0.013161,-0.574463
1,30d,0.007665,0.000081,0.009020,-0.804998


## 5. Évaluation sur les données de test

In [7]:
# Cibles réelles et prédictions — Test 7 jours
y_test_7d = test_7d["target_volatility_7d"]
y_pred_test_7d = test_7d["volatility_7d"]

# Cibles réelles et prédictions — Test 30 jours
y_test_30d = test_30d["target_volatility_30d"]
y_pred_test_30d = test_30d["volatility_30d"]

# Évaluation finale de la baseline
test_results = pd.DataFrame([
    evaluate_baseline(y_test_7d, y_pred_test_7d, "7d"),
    evaluate_baseline(y_test_30d, y_pred_test_30d, "30d")
])

display(test_results)

,horizon,MAE,MSE,RMSE,R2
0,7d,0.009354,0.000180,0.013431,-0.482664
1,30d,0.006931,0.000087,0.009353,-0.567560


## 6. Exporter les résultats

In [8]:
# Préparer les résultats de test pour l'export
baseline_results = test_results.copy()

baseline_results.insert(0, "tool", "Baseline")
baseline_results.insert(2, "model", "VolatilityPersistence")

# Conserver les métriques disponibles
baseline_results = baseline_results[
    ["tool", "horizon", "model", "MAE", "MSE", "RMSE", "R2"]
]

# Dossier d'export
export_dir = (
    project_root / "docs" / "benchmark" / "results_benchmark"
)
export_dir.mkdir(parents=True, exist_ok=True)

# Enregistrement
export_path = export_dir / "baseline_test_results.csv"

baseline_results.to_csv(export_path, index=False)

print("Export terminé :", export_path)
print("Dimensions :", baseline_results.shape)

display(baseline_results)

Export terminé : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\docs\benchmark\results_benchmark\baseline_test_results.csv
Dimensions : (2, 7)


,tool,horizon,model,MAE,MSE,RMSE,R2
0,Baseline,7d,VolatilityPersistence,0.009354,0.000180,0.013431,-0.482664
1,Baseline,30d,VolatilityPersistence,0.006931,0.000087,0.009353,-0.567560
